# QReCC IterCQR Baseline

This notebook evaluates the frozen TopiOCQA-trained IterCQR checkpoint on QReCC. The thesis calls the history-aware baseline at budget $B=512$ route $I_{512}$. Sparse/BM25 runs locally using the QReCC Lucene index. Dense/ANCE uses two successive sessions with disjoint local shard ranges.

### Local checkout setup

Use this public checkout on a local machine or a prepared Colab runtime. The next cell locates it; it does not clone a private repository, mount Drive, or request credentials.

In [ ]:
from pathlib import Path
import sys
import os

# Use an existing public checkout, locally or in Colab.
# No private repository, deploy key, or Drive mount is required.
ROOT = Path(os.environ.get("ROCC_ROOT", os.environ.get("MASTER_THESIS_PROJECT_ROOT", Path.cwd()))).resolve()
if not (ROOT / "experiments" / "rocc").exists():
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "experiments" / "rocc").exists())
os.environ["ROCC_ROOT"] = str(ROOT)
os.environ["MASTER_THESIS_PROJECT_ROOT"] = str(ROOT)
sys.path.insert(0, str(ROOT / "experiments"))
print("project:", ROOT)


In [2]:
!pip install -q pytorch-crf==0.7.2

### Runtime setup

Locates the `experiments` directory, exposes the ROCC API, and initializes the local or Colab runtime.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

EXPERIMENTS_DIR = ROOT / "experiments"
if str(EXPERIMENTS_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENTS_DIR))

import pandas as pd
import torch
from IPython.display import display

import rocc
from rocc.paths import project_path

pd.set_option("display.max_colwidth", 180)
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("experiments_dir:", EXPERIMENTS_DIR)
print("device:", DEVICE)


### IterCQR runtime resources

Reuses the local IterCQR checkpoint. In Colab, the helper can mount Google Drive and link a checkpoint found there when the local checkpoint is absent. This cell does not prepare the QReCC corpus.

In [4]:
rocc.prepare_colab_itercqr_checkpoint()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


ColabIterCQRCheckpointResult(is_colab=True, mounted=True, drive_mount=PosixPath('/content/drive'), drive_checkpoint=PosixPath('/content/drive/MyDrive/IterCQR'), local_checkpoint=PosixPath('<original-checkout>/experiments/model/IterCQR/IterCQR Model'), status='already linked', missing_files=(), size_bytes=892500921)

### Validate processed QReCC splits

These numeric-ID train/test splits and their manifest must already be local. Prepare them with notebook 00a or place the supplied prepared files at the documented paths.

In [ ]:
QRECC_DATA_DIR = project_path("experiments", "data", "qrecc")
QRECC_AZURE_SAS_FILE = None

qrecc_resources = rocc.resolve_qrecc_resources(QRECC_DATA_DIR)
qrecc_artifacts = rocc.inspect_qrecc_artifacts(qrecc_resources)
display(pd.DataFrame([qrecc_artifacts.to_dict()]))

qrecc_processed_splits_ready = all([
    qrecc_resources.processed_train_json.exists(),
    qrecc_resources.processed_test_json.exists(),
    qrecc_resources.collection_manifest_json.exists(),
])
print("processed_splits_ready:", qrecc_processed_splits_ready)

if not qrecc_processed_splits_ready:
    raise FileNotFoundError("Prepare QReCC numeric-ID splits with notebook 00a or copy the supplied collection metadata first.")

qrecc_artifacts = rocc.inspect_qrecc_artifacts(qrecc_resources)
qrecc_processed_splits_ready = all([
    qrecc_resources.processed_train_json.exists(),
    qrecc_resources.processed_test_json.exists(),
    qrecc_resources.collection_manifest_json.exists(),
])
print("processed_splits_ready:", qrecc_processed_splits_ready)
display(pd.DataFrame([qrecc_artifacts.to_dict()]))

data = rocc.load_itercqr_dataloaders(
    dataset_name="qrecc",
    splits=("test","train"),
    data_dir=QRECC_DATA_DIR,
    batch_size=8,
    progress=True,
    include_history=False,
)

test_loader = data.dataloader("test")
train_loader = data.dataloader("train")
first_batch = next(iter(test_loader))
qrecc_frame = data.frame

print("dataset:", data.dataset_name)
print("resource_root:", data.resources.root)
print("test_turns:", len(test_loader.dataset))
print("test_batches:", len(test_loader))
print("train_turns:", len(train_loader.dataset))
print("train_batches:", len(train_loader))
print("input_shape:", tuple(first_batch["bt_input_ids"].shape))

display(pd.DataFrame({
    "sample_id": first_batch["bt_sample_ids"],
    "question": first_batch["bt_questions"],
    "input_length": first_batch["bt_input_lengths"].tolist(),
}))


The gold-evaluable population contains **29,596 training turns** and **8,209 test turns**. With batch size 8, these form 3,700 and 1,027 batches. The first batch contains the shortest current-query-only inputs. Questions such as “why” and “how” demonstrate their dependence on dialogue history.

### IterCQR dataset diagnostics

`analyze_itercqr_dataset()` compares the complete processed QReCC splits with the gold-filtered turns present in the actual dataloaders. It reports conversation counts, batch counts, dynamic-padding savings, input-level truncation, and unknown-token frequencies.

The segment analysis distinguishes raw tokens, tokens retained by each segment limit, and tokens finally included under the global input budget. It therefore separates segment-level truncation from total-budget truncation.

In [6]:
analysis = rocc.analyze_itercqr_dataset(data, formatted_sample_size=10)

display(analysis.split_summary)
display(analysis.dataloader_summary)
display(analysis.padding_report)
display(analysis.truncation_report)
display(analysis.unknown_token_report)
display(analysis.segment_bucket_report)
display(analysis.segment_bucket_samples)
display(analysis.formatted_input_samples)


,split,conversations,turns,longest_conversation_turns,shortest_conversation_turns,mean_turns_per_conversation,turns_without_positive_ctx_passage_ids
0,all,13598,79952,12,1,5.88,42147
1,test,2775,16451,12,1,5.93,8242
2,train,10823,63501,12,1,5.87,33905


,split,num_turns,num_batches,batch_size,num_workers
0,test,8209,1027,8,0
1,train,29596,3700,8,0


,split,num_turns,batch_size,max_input_tokens,length_min,length_mean,length_median,length_max,real_tokens,static_total_tokens,static_pad_tokens,static_pad_fraction,bucketed_dynamic_total_tokens,bucketed_dynamic_pad_tokens,bucketed_dynamic_pad_fraction,padding_fraction_saved_points,padding_tokens_saved
0,test,8209,8,512,4,11.653064,11.0,32,95660,4203008,4107348,0.977240,95760,100,0.001044,0.976196,4107248
1,train,29596,8,512,4,11.781795,11.0,32,348694,15153152,14804458,0.976989,348808,114,0.000327,0.976662,14804344


,split,num_turns,max_input_tokens,raw_length_min,raw_length_mean,raw_length_median,raw_length_max,final_length_min,final_length_mean,final_length_median,final_length_max,truncated_turns,truncation_rate,truncated_tokens_total,mean_truncated_tokens_all,mean_truncated_tokens_truncated_only
0,test,8209,512,4,11.653064,11.0,32,4,11.653064,11.0,32,0,0.000000,0,0.00000,0.0
1,train,29596,512,4,11.783045,11.0,43,4,11.781795,11.0,32,5,0.000169,37,0.00125,7.4


,split,unk_token_id,unk_tokens,unk_rate,examples_with_unk,examples_with_unk_fraction
0,test,2,9,0.000094,7,0.000853
1,train,2,42,0.000120,31,0.001047


,split,segment_type,budget_tokens,count,raw_tokens_min,raw_tokens_mean,raw_tokens_max,bucket_tokens_min,bucket_tokens_mean,bucket_tokens_max,included_tokens_min,included_tokens_mean,included_tokens_max,bucket_utilization_mean,bucket_utilization_max,bucket_truncation_rate,global_truncation_rate
0,test,current_query,32,8209,4,11.653064,32,4,11.653064,32,4,11.653064,32,0.364158,1.0,0.000000,0.0
1,train,current_query,32,29596,4,11.783045,43,4,11.781795,32,4,11.781795,32,0.368181,1.0,0.000169,0.0


,split,sample_id,conv_id,turn_id,segment_index,segment_type,budget_tokens,raw_tokens,bucket_tokens,included_tokens,bucket_truncated_tokens,global_truncated_tokens,bucket_utilization,included_utilization
0,test,test:1:2,1,2,0,current_query,32,13,13,13,0,0,0.40625,0.40625
1,test,test:1:3,1,3,0,current_query,32,8,8,8,0,0,0.25000,0.25000
2,test,test:1:7,1,7,0,current_query,32,16,16,16,0,0,0.50000,0.50000
3,test,test:1:8,1,8,0,current_query,32,16,16,16,0,0,0.50000,0.50000
4,test,test:1:10,1,10,0,current_query,32,14,14,14,0,0,0.43750,0.43750
5,test,test:1:12,1,12,0,current_query,32,20,20,20,0,0,0.62500,0.62500
6,test,test:2:2,2,2,0,current_query,32,11,11,11,0,0,0.34375,0.34375
7,test,test:2:3,2,3,0,current_query,32,10,10,10,0,0,0.31250,0.31250
8,test,test:2:4,2,4,0,current_query,32,14,14,14,0,0,0.43750,0.43750
9,test,test:2:5,2,5,0,current_query,32,10,10,10,0,0,0.31250,0.31250


,split,sample_id,conv_id,turn_id,topic_document,topic_id,topic_switch,topic_switch_depth,topics_in_conversation,question,...,final_input_length,was_truncated,truncated_tokens,current_query_raw_tokens,current_query_bucket_tokens,history_segments_used,history_bucket_utilization_mean,history_bucket_utilization_max,formatted_input_text,positive_ctx_passage_ids
0,test,test:1:2,1,2,None,trec,False,None,1,What are the educational requirements required to become one?,...,13,False,0,13,13,0,0.0,0.0,question: What are the educational requirements required to become one?,[54355562]
1,test,test:1:3,1,3,None,trec,False,None,1,What does it cost?,...,8,False,0,8,8,0,0.0,0.0,question: What does it cost?,[54312465]
2,test,test:1:7,1,7,None,trec,False,None,1,What is the PA average salary vs an RN?,...,16,False,0,16,16,0,0.0,0.0,question: What is the PA average salary vs an RN?,[54367470]
3,test,test:1:8,1,8,None,trec,False,None,1,What the difference between a PA and a nurse practitioner?,...,16,False,0,16,16,0,0.0,0.0,question: What the difference between a PA and a nurse practitioner?,[54291674]
4,test,test:1:10,1,10,None,trec,False,None,1,Is a PA above a NP?,...,14,False,0,14,14,0,0.0,0.0,question: Is a PA above a NP?,[54311406]
5,test,test:1:12,1,12,None,trec,False,None,1,How much longer does it take to become a doctor after being an NP?,...,20,False,0,20,20,0,0.0,0.0,question: How much longer does it take to become a doctor after being an NP?,[54290087]
6,test,test:2:2,2,2,None,trec,False,None,1,Tell me about boer goats.,...,11,False,0,11,11,0,0.0,0.0,question: Tell me about boer goats.,"[18960438, 54470377]"
7,test,test:2:3,2,3,None,trec,False,None,1,What breed is good for meat?,...,10,False,0,10,10,0,0.0,0.0,question: What breed is good for meat?,[54283366]
8,test,test:2:4,2,4,None,trec,False,None,1,Are angora goats good for it?,...,14,False,0,14,14,0,0.0,0.0,question: Are angora goats good for it?,[54381343]
9,test,test:2:5,2,5,None,trec,False,None,1,What about boer goats?,...,10,False,0,10,10,0,0.0,0.0,question: What about boer goats?,[54283366]


The complete processed QReCC data contains **79,952 turns in 13,598 conversations**. Of these, **42,147 turns lack gold passage IDs** and are excluded from retrieval evaluation. The resulting dataloaders contain **29,596 training** and **8,209 test turns**.

Because `include_history=False`, every example contains only the current query. Mean input length is 11.78 tokens for Train and 11.65 for Test. Static padding to 512 tokens would waste about **97.7%** of each tensor. Length-bucketed dynamic padding reduces this to **0.03% on Train** and **0.10% on Test**.

No test query is truncated. Only **5 training queries** exceed the 32-token query limit. Unknown-token rates are approximately **0.01%** in both splits. The formatted examples confirm that no history segments are included and illustrate unresolved conversational references such as “one”, “it”, and “What about”.

### Validate local IterCQR assets

Provide the author checkpoint at the documented local model path. The original `t5-base` backbone can be downloaded on first use; inference uses the fine-tuned IterCQR checkpoint.

In [7]:
if rocc.is_google_colab():
    checkpoint_result = rocc.prepare_colab_itercqr_checkpoint()
    display(pd.DataFrame([checkpoint_result.to_dict()]))

t5_backbone_dir = rocc.ensure_hf_t5_backbone("t5-base")
itercqr_checkpoint_dir = rocc.validate_itercqr_checkpoint()

print("hf_t5_backbone_dir:", t5_backbone_dir)
print("itercqr_checkpoint_dir:", itercqr_checkpoint_dir)
print("itercqr_checkpoint_files:", rocc.iter_checkpoint_files())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,is_colab,mounted,drive_mount,drive_checkpoint,local_checkpoint,status,complete,missing_files,size_gib
0,True,True,/content/drive,/content/drive/MyDrive/IterCQR,/experiments/model/IterCQR/IterCQR Model,already linked,True,[],0.83


hf_t5_backbone_dir: <original-checkout>/experiments/model/hf/t5-base
itercqr_checkpoint_dir: <original-checkout>/experiments/model/IterCQR/IterCQR Model
itercqr_checkpoint_files: ['config.json', 'generation_config.json', 'pytorch_model.bin', 'special_tokens_map.json', 'spiece.model', 'tokenizer_config.json']


### IterCQR inference check

Loads the frozen IterCQR model and tokenizer on the selected device. One test batch is rewritten with deterministic greedy decoding and displayed alongside the original questions and gold passage IDs.

In [8]:
itercqr_rewriter = rocc.load_itercqr_rewriter(device=DEVICE)
print("model:", type(itercqr_rewriter.model).__name__)
print("tokenizer:", type(itercqr_rewriter.tokenizer).__name__)
print("device:", itercqr_rewriter.device)

rewrite_df = itercqr_rewriter.rewrite_dataloader(test_loader, max_batches=1)
display(rewrite_df[["sample_id", "question", "rewrite", "positive_ctx_passage_ids"]])


[transformers] The following generation flags are not valid and may be ignored: ['length_penalty']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


model: T5ForConditionalGeneration
tokenizer: T5Tokenizer
device: cuda:0


,sample_id,question,rewrite,positive_ctx_passage_ids
0,test:1531:2,why,"what is the reason for the existence of the company ""The Company"" in the early 1900s and early 1900s based in the United States and Europe",[54502782]
1,test:1903:2,How,How is the process of determining whether or not the person who is a member of the family of the person who is a member of the family,[54352243]
2,test:1903:5,Why,"What is the reason for the existence of the company ""The Company"" in the early 1900s and early 1900s based in the United States and Europe",[54570787]
3,test:1948:4,why,"what is the reason for the existence of the company ""The Company"" in the early 1900s and early 1900s based in the United States and Europe",[54308875]
4,test:1964:3,How,How is the process of determining whether or not the person who is a member of the family of the person who is a member of the family,[54522144]
5,test:1964:5,How,How is the process of determining whether or not the person who is a member of the family of the person who is a member of the family,[54522144]
6,test:2187:2,Why,"What is the reason for the existence of the company ""The Company"" in the early 1900s and early 1900s based in the United States and Europe",[54452536]
7,test:2343:4,why,"what is the reason for the existence of the company ""The Company"" in the early 1900s and early 1900s based in the United States and Europe",[54344699]


The eight QReCC turns collapse into only two generic rewrite patterns despite belonging to different conversations and gold passages. Once history is removed, the inputs “why” and “how” contain almost no discriminative information, so the decoder produces nearly identical outputs. This confirms that QReCC follow-up turns must be evaluated with their conversational context.

### Retrieval dependencies

Use Java 21 and Pyserini 2.0. The compatibility helper reports the installed environment; see the README for setup.

In [9]:
retrieval_deps = rocc.ensure_colab_retrieval_dependencies()
display(pd.DataFrame([retrieval_deps.to_dict()]))

,is_colab,installed,pyserini_available,java_home,java_available,packages,commands_run
0,True,False,True,/usr/lib/jvm/java-21-openjdk-amd64,True,[pyserini==2.0.0],[]


### QReCC BM25 runtime

Checks the Lucene index and its build status against the **54,573,064 source passages**. The thesis BM25 index contains **54,241,550 passages** after the empty-passage exclusions described in NB00a. The retriever uses $k_1=0.82$ and $b=0.68$. An incomplete or unavailable index prevents retrieval from starting.

In [10]:
QRECC_BM25_INDEX_DIR = project_path(
    "experiments",
    "data",
    "pyserini_bm25_lucene_qrecc",
    "lucene_index",
)
QRECC_BM25_LOG_FILE = project_path(
    "experiments",
    "logs",
    "qrecc_bm25_lucene_index.log",
)
QRECC_BM25_K1 = 0.82
QRECC_BM25_B = 0.68

qrecc_bm25_status = rocc.inspect_bm25_lucene_index(
    QRECC_BM25_INDEX_DIR,
    source_passages=rocc.QRECC_TOTAL_PASSAGES,
    log_file=QRECC_BM25_LOG_FILE,
)
display(pd.DataFrame([qrecc_bm25_status.to_dict()]))

if qrecc_bm25_status.loadable:
    bm25_retriever = rocc.load_bm25_retriever(
        index_dir=QRECC_BM25_INDEX_DIR,
        k1=QRECC_BM25_K1,
        b=QRECC_BM25_B,
    )
    print("QReCC BM25 Lucene index loaded:", bm25_retriever.index_dir)
else:
    bm25_retriever = None
    print("QReCC BM25 skipped: Lucene index is not local/loadable.")


,index_dir,exists,loadable,complete,num_docs,source_passages,log_file,log_exit_code,log_indexing_complete,log_indexed_docs,log_empty_docs,error
0,/experiments/data/pyserini_bm25_lucene_qrecc/lucene_index,False,False,False,None,54573064,/experiments/logs/qrecc_bm25_lucene_index.log,None,False,None,None,index directory does not exist


QReCC BM25 skipped: Lucene index is not local/loadable.


### BM25 smoke retrieval

Retrieves the top five BM25 passages for the first generated IterCQR rewrite. Raw passage content supports qualitative inspection, while min-max normalization makes scores comparable within the returned ranking.

In [11]:
if bm25_retriever is not None:
    query = rewrite_df.iloc[0]["rewrite"]
    bm25_hits = bm25_retriever.search(query, top_k=5, include_raw=True)
    bm25_hits_norm = rocc.normalize_retrieval_scores(bm25_hits, method="minmax")
    display(pd.DataFrame(bm25_hits_norm))
else:
    print("BM25 smoke retrieval skipped: bm25_retriever is not loaded.")


BM25 smoke retrieval skipped: bm25_retriever is not loaded.


The current-query-only rewrite drives BM25 toward passages containing generic terms such as “early 1900s” and “United States”. The highest-ranked passages are therefore lexically compatible but unrelated to the intended QReCC conversation. The expected gold passage is absent from the displayed top five, showing how a context-free generic rewrite propagates directly into retrieval failure. Normalized scores support only within-list comparison and do not alter the ranking.

### QReCC ANCE retrieval session

Configures the second of two retrieval sessions over the **54,573,064-passage** QReCC ANCE index. `session_2` processes shards 273–545, while `session_1` covers shards 0–272. Each session retains its top 100 results per query. The two partial ranking dumps are later merged by the unchanged inner-product scores $s_{\mathrm{ANCE}}(q,d)=\mathbf v_q^\top\mathbf v_d$ into the final global top 100. Place the complete index locally before running; `max_batches=None` enables retrieval over the complete evaluable test population.

In [ ]:
QRECC_ANCE_SESSION_ID = "session_2"  # session_1: shards 0..272, session_2: shards 273..545
QRECC_ANCE_RUN_ID = "qrecc_test_dense_itercqr"
QRECC_ANCE_PARTIAL_TOP_K = 100
QRECC_ANCE_FINAL_TOP_K = 100
QRECC_ANCE_RETRIEVAL_MAX_BATCHES = None

QRECC_ANCE_START_SHARD, QRECC_ANCE_END_SHARD = rocc.qrecc_ance_session_shard_range(QRECC_ANCE_SESSION_ID)
QRECC_ANCE_EXPECTED_START_PASSAGE = QRECC_ANCE_START_SHARD * rocc.QRECC_ANCE_SHARD_PASSAGES
QRECC_ANCE_EXPECTED_END_PASSAGE = min(
    (QRECC_ANCE_END_SHARD + 1) * rocc.QRECC_ANCE_SHARD_PASSAGES,
    rocc.QRECC_TOTAL_PASSAGES,
)

DENSE_INDEX_DIR = project_path(
    "experiments",
    "data",
    "pyserini_ance_faiss_qrecc",
    "faiss_flat_index_full_sharded",
)
DENSE_AZURE_SAS_FILE = None
DENSE_INDEX_DTYPE = "float32"
DENSE_CHUNK_SIZE = 50_000
DENSE_QUERY_BATCH_SIZE = 64

QRECC_ANCE_LOCAL_DUMP_DIR = project_path("experiments", "results", "qrecc", "ance_partial_runs")
QRECC_ANCE_OWN_DUMP = QRECC_ANCE_LOCAL_DUMP_DIR / QRECC_ANCE_RUN_ID / f"{QRECC_ANCE_SESSION_ID}.jsonl"
QRECC_ANCE_OTHER_SESSION_ID = "session_2" if QRECC_ANCE_SESSION_ID == "session_1" else "session_1"
QRECC_ANCE_OTHER_DUMP = QRECC_ANCE_LOCAL_DUMP_DIR / QRECC_ANCE_RUN_ID / f"{QRECC_ANCE_OTHER_SESSION_ID}.jsonl"
QRECC_ANCE_MERGED_DUMP = QRECC_ANCE_LOCAL_DUMP_DIR / QRECC_ANCE_RUN_ID / "merged.jsonl"

display(pd.DataFrame([{
    "session_id": QRECC_ANCE_SESSION_ID,
    "start_shard": QRECC_ANCE_START_SHARD,
    "end_shard": QRECC_ANCE_END_SHARD,
    "expected_start_passage": QRECC_ANCE_EXPECTED_START_PASSAGE,
    "expected_end_passage": QRECC_ANCE_EXPECTED_END_PASSAGE,
    "dense_index_dir": str(DENSE_INDEX_DIR),
    "own_dump": str(QRECC_ANCE_OWN_DUMP),
}]))


### Validate the local ANCE range

The local shards must match the assigned session, including shard count and passage boundaries. No automatic remote download occurs. The original GPU-memory check reserves 4 GiB before loading float32 vectors.

In [ ]:
def qrecc_dense_summary_matches_session(summary):
    return bool(
        summary.shard_count == QRECC_ANCE_END_SHARD - QRECC_ANCE_START_SHARD + 1
        and summary.first_passage_start == QRECC_ANCE_EXPECTED_START_PASSAGE
        and summary.last_passage_end == QRECC_ANCE_EXPECTED_END_PASSAGE
    )


try:
    dense_summary = rocc.summarize_dense_faiss_shards(
        DENSE_INDEX_DIR,
        start_shard=QRECC_ANCE_START_SHARD,
        end_shard=QRECC_ANCE_END_SHARD,
    )
    DENSE_INDEX_DIR = dense_summary.index_dir
except (FileNotFoundError, RuntimeError, ValueError, KeyError):
    dense_summary = None

if dense_summary is not None and not qrecc_dense_summary_matches_session(dense_summary):
    print("QReCC local ANCE shard range does not match this session; finish the local copy/build first.")
    display(pd.DataFrame([dense_summary.to_dict()]))
    dense_summary = None

if dense_summary is None:
    print("Local ANCE shard range is missing/incomplete; finish copying or building it first.")

if dense_summary is not None:
    DENSE_INDEX_DIR = dense_summary.index_dir
    display(pd.DataFrame([dense_summary.to_dict()]))

    if DEVICE.startswith("cuda"):
        dense_capacity = rocc.check_dense_gpu_capacity(
            dense_summary,
            device=DEVICE,
            index_dtype=DENSE_INDEX_DTYPE,
            reserve_gib=4.0,
        )
        display(pd.DataFrame([dense_capacity.__dict__]))
    else:
        dense_capacity = None
        print("CUDA is not active; QReCC ANCE Torch search is requires a sufficiently large CUDA device.")
else:
    dense_summary = None
    dense_capacity = None


### QReCC partial ANCE retriever

Loads only the ANCE shard range assigned to the active QReCC session. Loading requires CUDA, sufficient GPU memory, and a complete local shard range. A five-passage search for “Eliza Fletcher” verifies operation within the partial index. Loading progress is written to a session-specific log. Final rankings require merging the results from both QReCC sessions.

In [ ]:
dense_retriever = None

if DENSE_INDEX_DIR.exists() and DEVICE.startswith("cuda") and dense_capacity is not None and dense_capacity.fits:
    dense_retriever = rocc.load_dense_ance_torch_retriever(
        index_dir=DENSE_INDEX_DIR,
        device=DEVICE,
        index_dtype=DENSE_INDEX_DTYPE,
        start_shard=QRECC_ANCE_START_SHARD,
        end_shard=QRECC_ANCE_END_SHARD,
        chunk_size=DENSE_CHUNK_SIZE,
        query_batch_size=DENSE_QUERY_BATCH_SIZE,
        progress=True,
        status_log=project_path("experiments", "logs", f"ance_index_load_status_{QRECC_ANCE_SESSION_ID}.log"),
    )
    display(pd.DataFrame(dense_retriever.search("Eliza Fletcher", top_k=5)))
else:
    print("QReCC dense retriever not loaded: requires a local ANCE shard range, CUDA, and sufficient free VRAM.")


### QReCC B512 baseline configuration

Evaluates all **8,209 gold-evaluable QReCC test queries** with route $I_{512}$. The current query $q_t$ comes first, followed by historical answers and queries from $H_t$ in most-recent-first order. Each field is limited to 32 T5 tokens before the total input budget $B=512$ is applied, so the serialized input length satisfies $n\leq B$. Prefixes and special tokens count towards these limits. Greedy decoding has maximum output sequence length $m_{\max}=32$.

Retrieval depth is $k_{\mathrm{ret}}=100$. The reported metrics are $\mathrm{MRR}$ over those retrieved passages, $\mathrm{nDCG}@3$, $\mathrm{Recall}@10$, and $\mathrm{Recall}@100$. Their 95% confidence intervals use $N_{\mathrm{boot}}=10{,}000$ query-level bootstrap resamples. The later final retrieval experiments use depth 1,000 instead.

In [15]:
RETRIEVAL_TOP_K = 100
RETRIEVAL_MAX_BATCHES = None
EVAL_KS = (3, 10, 100)
BOOTSTRAP_SAMPLES = 10000
INCLUDE_HISTORY = True
QRECC_BASELINE_BUDGETS = (512,)

print("expected test queries:", len(test_loader.dataset))
print("include_history:", INCLUDE_HISTORY)
print("baseline-budgets:", QRECC_BASELINE_BUDGETS)

expected test queries: 8209
include_history: True
baseline-budgets: (512,)


### QReCC BM25 B512 baseline

Runs the canonical IterCQR baseline over all 8,209 gold-evaluable QReCC test queries. Recency-ordered dialogue history is included within the global 512-token input budget. Each rewrite is evaluated against BM25 using \(k_1=0.82\), \(b=0.68\), and the top 100 passages. Aggregate metrics, bootstrap confidence intervals, and per-query results are persisted under `full_pipeline_runs/qrecc_test_sparse_itercqr_max512`.

In [13]:
qrecc_sparse_budget_sweep = None
if bm25_retriever is not None:
    qrecc_sparse_budget_sweep = rocc.run_itercqr_budget_sweep(
        dataset_name="qrecc",
        split="test",
        retriever_kind="sparse",
        budgets=QRECC_BASELINE_BUDGETS,
        data_dir=QRECC_DATA_DIR,
        bm25_index_dir=QRECC_BM25_INDEX_DIR,
        azure_sas_file=QRECC_AZURE_SAS_FILE,
        run_id="qrecc_test_sparse_itercqr",
        batch_size=8,
        top_k=RETRIEVAL_TOP_K,
        eval_ks=EVAL_KS,
        bootstrap_samples=BOOTSTRAP_SAMPLES,
        max_batches=RETRIEVAL_MAX_BATCHES,
        device=DEVICE,
        progress=True,
        rewriter=itercqr_rewriter,
        retriever=bm25_retriever,
        include_history=INCLUDE_HISTORY,
        bm25_k1=QRECC_BM25_K1,
        bm25_b=QRECC_BM25_B,
    )
    display(qrecc_sparse_budget_sweep.summary)
    display(qrecc_sparse_budget_sweep.evaluation_aggregate)
else:
    print("Sparse budget sweep skipped: bm25_retriever is not loaded.")

[1/4] Prepare shared dataset resources
[2/4] Use shared IterCQR rewriter
[3/4] Use shared retriever index
[4/4] Use shared retriever
[1/4] Load dataset split for budget


[2/4] Run rewrite + retrieval


[3/4] Evaluate retrieval
[4/4] Save eval results


,budget_tokens,include_history,run_id,dataset_name,split,retriever_kind,qrecc_dense_mode,MRR,nDCG@3,R@3,nDCG@10,R@10,nDCG@100,R@100
0,512,True,qrecc_test_sparse_itercqr_max512,qrecc,test,sparse,run_partial,0.448366,0.423299,0.476625,0.475476,0.62237,0.522547,0.834477


,budget_tokens,include_history,metric,n,mean,variance
0,512,True,MRR,8209,0.448366,0.180825
1,512,True,nDCG@3,8209,0.423299,0.201307
2,512,True,R@3,8209,0.476625,0.239805
3,512,True,nDCG@10,8209,0.475476,0.174279
4,512,True,R@10,8209,0.622370,0.223956
5,512,True,nDCG@100,8209,0.522547,0.140991
6,512,True,R@100,8209,0.834477,0.130918


### QReCC ANCE B512 partial retrieval

Runs the canonical IterCQR B512 baseline over all 8,209 gold-evaluable QReCC test queries using the ANCE shard range assigned to the active session. Recency-ordered dialogue history is included within the global 512-token input budget. Each session retains its top 100 passages per query and saves the partial ranking dump locally. Final metrics are calculated only after merging the two session outputs.

In [16]:
qrecc_dense_partial_budget_sweep = None
if dense_retriever is not None:
    qrecc_dense_partial_budget_sweep = rocc.run_itercqr_budget_sweep(
        dataset_name="qrecc",
        split="test",
        retriever_kind="dense",
        budgets=QRECC_BASELINE_BUDGETS,
        data_dir=QRECC_DATA_DIR,
        dense_index_dir=DENSE_INDEX_DIR,
        azure_sas_file=DENSE_AZURE_SAS_FILE,
        qrecc_session_id=QRECC_ANCE_SESSION_ID,
        qrecc_dense_mode="run_partial",
        run_id=QRECC_ANCE_RUN_ID,
        batch_size=8,
        top_k=QRECC_ANCE_PARTIAL_TOP_K,
        eval_ks=EVAL_KS,
        bootstrap_samples=BOOTSTRAP_SAMPLES,
        max_batches=QRECC_ANCE_RETRIEVAL_MAX_BATCHES,
        device=DEVICE,
        progress=True,
        rewriter=itercqr_rewriter,
        retriever=dense_retriever,
        include_history=INCLUDE_HISTORY,
    )
    display(qrecc_dense_partial_budget_sweep.summary)
else:
    print("Dense partial budget sweep skipped: dense_retriever is not loaded.")

[1/4] Prepare shared dataset resources
[2/4] Use shared IterCQR rewriter
[3/4] Use shared retriever index
[4/4] Use shared retriever
[1/4] Load dataset split for budget


[2/4] Run rewrite + retrieval


[3/4] Export QReCC partial dump
[4/4] Upload QReCC partial dump when SAS is available
upload QReCC partial dense dump to Azure: <private-storage-omitted>


,budget_tokens,include_history,run_id,dataset_name,split,retriever_kind,qrecc_dense_mode,partial_dump_file,partial_dump_queries,partial_dump_hits
0,512,True,qrecc_test_ance_itercqr_max512,qrecc,test,dense,run_partial,/content/qrecc_ance_partial_dumps/qrecc_test_ance_itercqr_max512/session_2.jsonl,8209,820900


### QReCC dense merge and evaluation

Reads the local `session_1` and `session_2` ANCE partial rankings for the configured run once both exist. Both shard-specific top-100 lists are merged query-wise by their raw ANCE scores, reranked globally, and truncated to the final top 100 passages.

The merged rankings are evaluated over all 8,209 gold-evaluable QReCC test queries. Aggregate retrieval metrics, per-query results, and bootstrap confidence intervals are stored under the corresponding `full_pipeline_runs` directory.

In [ ]:
RUN_QRECC_DENSE_BUDGET_MERGE_EVAL = True

qrecc_dense_merge_budget_sweep = None
if (
    RUN_QRECC_DENSE_BUDGET_MERGE_EVAL
    and all((QRECC_ANCE_LOCAL_DUMP_DIR / f"{QRECC_ANCE_RUN_ID}_max{budget}" / f"{session}.jsonl").is_file()
            for budget in QRECC_BASELINE_BUDGETS for session in ("session_1", "session_2"))
):
    qrecc_dense_merge_budget_sweep = rocc.run_itercqr_budget_sweep(
        dataset_name="qrecc",
        split="test",
        retriever_kind="dense",
        budgets=QRECC_BASELINE_BUDGETS,
        data_dir=QRECC_DATA_DIR,
        dense_index_dir=DENSE_INDEX_DIR,
        azure_sas_file=DENSE_AZURE_SAS_FILE,
        qrecc_dense_mode="merge_eval",
        run_id=QRECC_ANCE_RUN_ID,
        batch_size=8,
        top_k=QRECC_ANCE_FINAL_TOP_K,
        eval_ks=EVAL_KS,
        bootstrap_samples=BOOTSTRAP_SAMPLES,
        device=DEVICE,
        progress=True,
        include_history=INCLUDE_HISTORY,
    )
    display(qrecc_dense_merge_budget_sweep.summary)
    display(
        qrecc_dense_merge_budget_sweep.evaluation_aggregate
    )
else:
    print("Dense merge/evaluation requires both local session dumps. Run session_1 and session_2 with the same run ID first.")


### BM25 bootstrap summary

Reports bootstrap estimates and 95% confidence intervals for the QReCC BM25 $I_{512}$ baseline. The summary is available after the sparse baseline run has completed successfully.

In [15]:
if qrecc_sparse_budget_sweep is not None:
    display(qrecc_sparse_budget_sweep.bootstrap_summary)
else:
    print("Sparse evaluation summary skipped: qrecc_sparse_budget_sweep is unavailable.")

,budget_tokens,include_history,metric,bootstrap_samples,confidence,mean,variance,ci95_low,ci95_high
0,512,True,MRR,10000,0.95,0.448394,0.000022,0.439306,0.457425
1,512,True,nDCG@3,10000,0.95,0.423335,0.000024,0.413749,0.432936
2,512,True,R@3,10000,0.95,0.476662,0.000029,0.466158,0.487092
3,512,True,nDCG@10,10000,0.95,0.475510,0.000021,0.466488,0.484545
4,512,True,R@10,10000,0.95,0.622389,0.000027,0.612239,0.632538
5,512,True,nDCG@100,10000,0.95,0.522580,0.000017,0.514665,0.530633
6,512,True,R@100,10000,0.95,0.834499,0.000015,0.826917,0.842191


### ANCE bootstrap summary

Reports bootstrap estimates and 95% confidence intervals for the QReCC ANCE $I_{512}$ baseline. The summary is available after both dense sessions and their final merge have completed successfully.

In [ ]:
if qrecc_dense_merge_budget_sweep is not None:
    display(qrecc_dense_merge_budget_sweep.bootstrap_summary)
else:
    print("Dense local budget merge/evaluation summary skipped.")
